In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.2 MB/s eta 0:00:00


In [2]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [9]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2026-06-30 13:24:32,404] A new study created in memory with name: no-name-c28d0e61-ea34-450b-b9a0-35d34c8b3dd8
[I 2026-06-30 13:24:33,051] Trial 0 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 68, 'max_depth': 5}. Best is trial 0 with value: 0.7746741154562384.
[I 2026-06-30 13:24:33,726] Trial 1 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 74, 'max_depth': 19}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-06-30 13:24:35,544] Trial 2 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 199, 'max_depth': 11}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-06-30 13:24:36,149] Trial 3 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 114, 'max_depth': 12}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-06-30 13:24:36,870] Trial 4 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 142, 'max_depth': 14}. Best is trial 1 with value: 0.7765363

In [10]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 125, 'max_depth': 19}


In [11]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


Random search Sampler


In [12]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-06-30 13:26:08,977] A new study created in memory with name: no-name-26c7bc8c-fc7a-486e-99ec-2244bd5a9c67
[I 2026-06-30 13:26:10,593] Trial 0 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 173, 'max_depth': 8}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-06-30 13:26:11,461] Trial 1 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 90, 'max_depth': 16}. Best is trial 1 with value: 0.7653631284916201.
[I 2026-06-30 13:26:12,829] Trial 2 finished with value: 0.7746741154562383 and parameters: {'n_estimators': 133, 'max_depth': 15}. Best is trial 2 with value: 0.7746741154562383.
[I 2026-06-30 13:26:13,439] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 129, 'max_depth': 6}. Best is trial 2 with value: 0.7746741154562383.
[I 2026-06-30 13:26:13,807] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 69, 'max_depth': 16}. Best is trial 2 with value: 0.77467411

In [13]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7783985102420857
Best hyperparameters: {'n_estimators': 63, 'max_depth': 13}


In [14]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


Grid search sampler

In [15]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [16]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-06-30 13:27:20,966] A new study created in memory with name: no-name-7ca8b619-493f-4dca-b87d-a4438ace2ca1
[I 2026-06-30 13:27:21,575] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-30 13:27:22,735] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-30 13:27:23,146] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-06-30 13:27:23,914] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-06-30 13:27:24,414] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [17]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [18]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


Optuna visualization


In [19]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [20]:
# 1. Optimization History
plot_optimization_history(study).show()

In [21]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [22]:
# 3. Slice Plot
plot_slice(study).show()

In [23]:
# 4. Contour Plot
plot_contour(study).show()

In [24]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

## Optimizing Multiple ML Models

In [25]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [26]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [27]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-06-30 13:30:59,487] A new study created in memory with name: no-name-3de454de-50bb-4756-9818-7a9a09af528f
[I 2026-06-30 13:31:00,609] Trial 0 finished with value: 0.761638733705773 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 114, 'learning_rate': 0.15527926535347586, 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 3}. Best is trial 0 with value: 0.761638733705773.
[I 2026-06-30 13:31:03,780] Trial 1 finished with value: 0.750465549348231 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 162, 'learning_rate': 0.07719376998640369, 'max_depth': 9, 'min_samples_split': 5, 'min_samples_leaf': 7}. Best is trial 0 with value: 0.761638733705773.
[I 2026-06-30 13:31:08,026] Trial 2 finished with value: 0.7355679702048418 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 208, 'learning_rate': 0.09992291824384114, 'max_depth': 16, 'min_samples_split': 4, 'min_samples_leaf': 7}. Best is trial 0 with value: 0.761638733705

In [28]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.1439145425667094, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [29]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.761639,2026-06-30 13:30:59.488663,2026-06-30 13:31:00.609067,0 days 00:00:01.120404,NaN,NaN,GradientBoosting,NaN,NaN,0.155279,3.0,3.0,10.0,114.0,COMPLETE
1,1,0.750466,2026-06-30 13:31:00.610521,2026-06-30 13:31:03.780620,0 days 00:00:03.170099,NaN,NaN,GradientBoosting,NaN,NaN,0.077194,9.0,7.0,5.0,162.0,COMPLETE
2,2,0.735568,2026-06-30 13:31:03.783418,2026-06-30 13:31:08.026674,0 days 00:00:04.243256,NaN,NaN,GradientBoosting,NaN,NaN,0.099923,16.0,7.0,4.0,208.0,COMPLETE
3,3,0.744879,2026-06-30 13:31:08.028008,2026-06-30 13:31:09.897248,0 days 00:00:01.869240,NaN,NaN,GradientBoosting,NaN,NaN,0.026921,9.0,4.0,7.0,110.0,COMPLETE
4,4,0.785847,2026-06-30 13:31:09.899644,2026-06-30 13:31:10.001880,0 days 00:00:00.102236,12.969653,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.787709,2026-06-30 13:31:39.395705,2026-06-30 13:31:39.417635,0 days 00:00:00.021930,0.172465,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.713222,2026-06-30 13:31:39.418301,2026-06-30 13:31:39.442087,0 days 00:00:00.023786,0.144967,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.785847,2026-06-30 13:31:39.442754,2026-06-30 13:31:39.464384,0 days 00:00:00.021630,0.274365,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.789572,2026-06-30 13:31:39.465115,2026-06-30 13:31:39.487178,0 days 00:00:00.022063,0.115948,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [30]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,78
GradientBoosting,13
RandomForest,9


In [31]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.736714
RandomForest,0.764329
SVM,0.775939


In [32]:
# 1. Optimization History
plot_optimization_history(study).show()

In [33]:
# 3. Slice Plot
plot_slice(study).show()

In [34]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [36]:
! pip install optuna-integration[xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 5.0 MB/s eta 0:00:00


In [41]:
# ==========================================================
# Optuna + XGBoost Hyperparameter Tuning Example
# Dataset : Iris
# ==========================================================

# Install Optuna integration (Run once)
!pip install -q optuna-integration[xgboost]

# -------------------------
# Import Libraries
# -------------------------
import optuna
import xgboost as xgb
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ----------------------------------------------------------
# STEP 1 : Load Dataset
# ----------------------------------------------------------
# X -> Features
# y -> Labels (0,1,2)

X, y = load_iris(return_X_y=True)

print("Dataset Shape :", X.shape)

# ----------------------------------------------------------
# STEP 2 : Split Dataset
# ----------------------------------------------------------
# 80% Training
# 20% Testing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ----------------------------------------------------------
# STEP 3 : Objective Function
# ----------------------------------------------------------
# Optuna calls this function once for every trial.
# Every trial uses DIFFERENT hyperparameters.

def objective(trial):

    # ------------------------------------------------------
    # Hyperparameter Search Space
    # Optuna will automatically choose values
    # ------------------------------------------------------

    params = {

        # Suppress training logs
        "verbosity": 0,

        # Multi-class classification
        "objective": "multi:softprob",

        # Number of classes
        "num_class": 3,

        # Validation metric
        "eval_metric": "mlogloss",

        # Tree based booster
        "booster": "gbtree",

        # L2 Regularization
        "lambda": trial.suggest_float(
            "lambda",
            1e-8,
            1.0,
            log=True
        ),

        # L1 Regularization
        "alpha": trial.suggest_float(
            "alpha",
            1e-8,
            1.0,
            log=True
        ),

        # Learning Rate
        "eta": trial.suggest_float(
            "eta",
            0.01,
            0.30
        ),

        # Minimum loss reduction
        "gamma": trial.suggest_float(
            "gamma",
            1e-8,
            1.0,
            log=True
        ),

        # Maximum depth of tree
        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            9
        ),

        # Minimum child weight
        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        # Fraction of rows used
        "subsample": trial.suggest_float(
            "subsample",
            0.4,
            1.0
        ),

        # Fraction of columns used
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.4,
            1.0
        )
    }

    # ------------------------------------------------------
    # Convert Dataset to DMatrix
    # DMatrix is XGBoost's optimized data structure
    # ------------------------------------------------------

    dtrain = xgb.DMatrix(
        X_train,
        label=y_train
    )

    dtest = xgb.DMatrix(
        X_test,
        label=y_test
    )

    # ------------------------------------------------------
    # Pruning Callback
    # Stops bad trials early
    # ------------------------------------------------------

    pruning_callback = optuna.integration.XGBoostPruningCallback(
        trial,
        "eval-mlogloss"
    )

    # ------------------------------------------------------
    # Train XGBoost
    # ------------------------------------------------------

    model = xgb.train(

        params,

        dtrain,

        # Maximum number of trees
        num_boost_round=300,

        # Evaluation datasets
        evals=[
            (dtrain, "train"),
            (dtest, "eval")
        ],

        # Stop if validation loss
        # doesn't improve for 30 rounds
        early_stopping_rounds=30,

        # Enable pruning
        callbacks=[pruning_callback]
    )

    # ------------------------------------------------------
    # Prediction
    # ------------------------------------------------------

    predictions = model.predict(dtest)

    # Convert probabilities to class labels
    predicted_labels = np.argmax(
        predictions,
        axis=1
    )

    # ------------------------------------------------------
    # Accuracy
    # ------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        predicted_labels
    )

    # Return accuracy to Optuna
    return accuracy


# ----------------------------------------------------------
# STEP 4 : Create Study
# ----------------------------------------------------------
# maximize -> Highest Accuracy
# Pruner -> Stops poor trials early

study = optuna.create_study(

    direction="maximize",

    pruner=optuna.pruners.SuccessiveHalvingPruner()
)

# ----------------------------------------------------------
# STEP 5 : Start Optimization
# ----------------------------------------------------------

study.optimize(
    objective,
    n_trials=50
)

# ----------------------------------------------------------
# STEP 6 : Best Result
# ----------------------------------------------------------

print("\n===============================")
print("BEST HYPERPARAMETERS")
print("===============================")

for key, value in study.best_trial.params.items():
    print(f"{key:20} : {value}")

print("\n===============================")
print("BEST ACCURACY")
print("===============================")

print(study.best_value)

[I 2026-06-30 13:48:10,274] A new study created in memory with name: no-name-4a55af2f-c471-43f8-a348-d7d00b1cd536


Dataset Shape : (150, 4)
[0]	train-mlogloss:1.00193	eval-mlogloss:1.01011
[1]	train-mlogloss:0.80266	eval-mlogloss:0.79417
[2]	train-mlogloss:0.65897	eval-mlogloss:0.63540
[3]	train-mlogloss:0.57459	eval-mlogloss:0.53715
[4]	train-mlogloss:0.49990	eval-mlogloss:0.45821
[5]	train-mlogloss:0.42708	eval-mlogloss:0.38136
[6]	train-mlogloss:0.38186	eval-mlogloss:0.33566
[7]	train-mlogloss:0.35470	eval-mlogloss:0.31750
[8]	train-mlogloss:0.31543	eval-mlogloss:0.27087
[9]	train-mlogloss:0.28803	eval-mlogloss:0.24258
[10]	train-mlogloss:0.27376	eval-mlogloss:0.22281
[11]	train-mlogloss:0.26476	eval-mlogloss:0.21091
[12]	train-mlogloss:0.25994	eval-mlogloss:0.20852
[13]	train-mlogloss:0.25213	eval-mlogloss:0.19840
[14]	train-mlogloss:0.25182	eval-mlogloss:0.19735
[15]	train-mlogloss:0.25107	eval-mlogloss:0.19649
[16]	train-mlogloss:0.24168	eval-mlogloss:0.18334
[17]	train-mlogloss:0.24148	eval-mlogloss:0.18231
[18]	train-mlogloss:0.24143	eval-mlogloss:0.18226
[19]	train-mlogloss:0.24083	eval-ml

[I 2026-06-30 13:48:11,230] Trial 0 finished with value: 1.0 and parameters: {'lambda': 5.983901338203304e-08, 'alpha': 9.1113450684657e-06, 'eta': 0.1940290447101199, 'gamma': 5.809704739686863e-05, 'max_depth': 4, 'min_child_weight': 6, 'subsample': 0.5320212572787584, 'colsample_bytree': 0.4497509937016343}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:0.90085	eval-mlogloss:0.88696
[1]	train-mlogloss:0.75400	eval-mlogloss:0.72758
[2]	train-mlogloss:0.63727	eval-mlogloss:0.60413
[3]	train-mlogloss:0.54317	eval-mlogloss:0.50744
[4]	train-mlogloss:0.46964	eval-mlogloss:0.43119
[5]	train-mlogloss:0.40834	eval-mlogloss:0.36358
[6]	train-mlogloss:0.35882	eval-mlogloss:0.31198
[7]	train-mlogloss:0.31927	eval-mlogloss:0.27047
[8]	train-mlogloss:0.28771	eval-mlogloss:0.23413
[9]	train-mlogloss:0.26351	eval-mlogloss:0.20905
[10]	train-mlogloss:0.24449	eval-mlogloss:0.18898
[11]	train-mlogloss:0.23031	eval-mlogloss:0.17258
[12]	train-mlogloss:0.21659	eval-mlogloss:0.15604
[13]	train-mlogloss:0.21159	eval-mlogloss:0.14913
[14]	train-mlogloss:0.20839	eval-mlogloss:0.14550
[15]	train-mlogloss:0.20705	eval-mlogloss:0.14380
[16]	train-mlogloss:0.20585	eval-mlogloss:0.14210
[17]	train-mlogloss:0.20410	eval-mlogloss:0.14014
[18]	train-mlogloss:0.20329	eval-mlogloss:0.14133
[19]	train-mlogloss:0.20160	eval-mlogloss:0.13921
[20]	train

[I 2026-06-30 13:48:11,691] Trial 1 finished with value: 1.0 and parameters: {'lambda': 2.5356403236317053e-07, 'alpha': 0.02734270886469923, 'eta': 0.16149675429930996, 'gamma': 0.0001776204519139857, 'max_depth': 6, 'min_child_weight': 8, 'subsample': 0.8169598095746593, 'colsample_bytree': 0.7659033867137852}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:0.79746	eval-mlogloss:0.77722
[1]	train-mlogloss:0.66181	eval-mlogloss:0.61967


[I 2026-06-30 13:48:16,035] Trial 48 pruned. Trial was pruned at iteration 2.


[0]	train-mlogloss:1.03256	eval-mlogloss:1.03625
[1]	train-mlogloss:0.95657	eval-mlogloss:0.95915
[2]	train-mlogloss:0.88817	eval-mlogloss:0.88874
[3]	train-mlogloss:0.82717	eval-mlogloss:0.82625
[4]	train-mlogloss:0.77190	eval-mlogloss:0.76818
[5]	train-mlogloss:0.72048	eval-mlogloss:0.71425
[6]	train-mlogloss:0.67491	eval-mlogloss:0.66575
[7]	train-mlogloss:0.63318	eval-mlogloss:0.62191


[I 2026-06-30 13:48:16,079] Trial 49 pruned. Trial was pruned at iteration 8.



BEST HYPERPARAMETERS
lambda               : 5.983901338203304e-08
alpha                : 9.1113450684657e-06
eta                  : 0.1940290447101199
gamma                : 5.809704739686863e-05
max_depth            : 4
min_child_weight     : 6
subsample            : 0.5320212572787584
colsample_bytree     : 0.4497509937016343

BEST ACCURACY
1.0


In [42]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()